In [1]:
import pennylane as qml
from pennylane import numpy as np
from pennylane.optimize import AdamOptimizer

from sklearn.model_selection import train_test_split
import pandas as pd

from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score

import math
import pickle
import os
# preparaing data
df = pd.read_csv('/home/lesc-neuro-covid/Felipe/quantumAlgorithms/QML/data/ML-EdgeIIoT-dataset.csv')
df.head()

/tmp/ipykernel_3137063/57914899.py:17: DtypeWarning: Columns (3,6,11,13,14,15,16,17,31,32,34,39,45,51,54,55) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/home/lesc-neuro-covid/Felipe/quantumAlgorithms/QML/data/ML-EdgeIIoT-dataset.csv')


,frame.time,ip.src_host,ip.dst_host,arp.dst.proto_ipv4,arp.opcode,arp.hw.size,arp.src.proto_ipv4,icmp.checksum,icmp.seq_le,icmp.transmit_timestamp,...,mqtt.proto_len,mqtt.protoname,mqtt.topic,mqtt.topic_len,mqtt.ver,mbtcp.len,mbtcp.trans_id,mbtcp.unit_id,Attack_label,Attack_type
0,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
1,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
2,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
3,6.0,192.168.0.101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM
4,6.0,192.168.0.152,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,MITM


In [2]:
df.dtypes

frame.time             object
ip.src_host            object
ip.dst_host            object
arp.dst.proto_ipv4     object
arp.opcode            float64
                       ...   
mbtcp.len             float64
mbtcp.trans_id        float64
mbtcp.unit_id         float64
Attack_label            int64
Attack_type            object
Length: 63, dtype: object

In [4]:
numerical_col = []
categorical_col = []

for i in df.columns:
    if df[i].nunique() == 1: 
        df.drop(i , axis =1 , inplace = True)
        print("dropped column : ", i)
    
    elif i != 'Attack_label' or i != 'Attack_type':
        if df[i].dtype == object or df[i].nunique() <= 20:
            categorical_col.append(i)
        else:
            numerical_col.append(i)

traget_col = ['Attack_label', 'Attack_type']
print(f"Colunas target: {traget_col}") 
print(f"Colunas categóricas: {categorical_col}")
print(f"Colunas numéricas {numerical_col}")

dropped column :  icmp.unused
dropped column :  http.tls_port
dropped column :  dns.qry.type
dropped column :  dns.retransmit_request_in
dropped column :  mqtt.msg_decoded_as
dropped column :  mbtcp.len
dropped column :  mbtcp.trans_id
dropped column :  mbtcp.unit_id
Colunas target: ['Attack_label', 'Attack_type']
Colunas categóricas: ['frame.time', 'ip.src_host', 'ip.dst_host', 'arp.dst.proto_ipv4', 'arp.opcode', 'arp.hw.size', 'arp.src.proto_ipv4', 'http.file_data', 'http.request.uri.query', 'http.request.method', 'http.referer', 'http.request.full_uri', 'http.request.version', 'http.response', 'tcp.connection.fin', 'tcp.connection.rst', 'tcp.connection.syn', 'tcp.connection.synack', 'tcp.flags', 'tcp.flags.ack', 'tcp.options', 'tcp.payload', 'tcp.srcport', 'dns.qry.name.len', 'dns.retransmission', 'dns.retransmit_request', 'mqtt.conack.flags', 'mqtt.conflag.cleansess', 'mqtt.conflags', 'mqtt.hdrflags', 'mqtt.len', 'mqtt.msg', 'mqtt.msgtype', 'mqtt.proto_len', 'mqtt.protoname', 'mqtt

In [5]:
zero_percent_columns = []

for col in df.columns:
    try:
        zero_ratio = (df[col] == 0).sum() / len(df)
        if zero_ratio >= 0.50:
            zero_percent_columns.append((col, zero_ratio))
            df.drop(col , axis=1 , inplace=True)
    except:
        # Skip non-numeric or problematic columns
        continue

# Display the results
for col, ratio in zero_percent_columns:
    print(f"{col}: {ratio:.2%} zeros")

print("number of dropped columns: " ,len(zero_percent_columns))
print("New DF Shape: ", df.shape)

arp.opcode: 99.00% zeros
arp.hw.size: 99.00% zeros
icmp.checksum: 90.70% zeros
icmp.seq_le: 90.08% zeros
icmp.transmit_timestamp: 99.95% zeros
http.content_length: 94.01% zeros
http.request.uri.query: 58.47% zeros
http.referer: 68.85% zeros
http.response: 95.43% zeros
tcp.connection.fin: 94.19% zeros
tcp.connection.rst: 90.59% zeros
tcp.connection.syn: 87.21% zeros
tcp.connection.synack: 97.01% zeros
tcp.len: 73.19% zeros
udp.port: 99.74% zeros
udp.stream: 90.79% zeros
udp.time_delta: 99.74% zeros
dns.qry.name: 98.95% zeros
dns.qry.name.len: 89.47% zeros
dns.qry.qu: 99.84% zeros
dns.retransmission: 99.98% zeros
dns.retransmit_request: 100.00% zeros
mqtt.conack.flags: 79.23% zeros
mqtt.conflag.cleansess: 99.21% zeros
mqtt.conflags: 99.21% zeros
mqtt.hdrflags: 96.79% zeros
mqtt.len: 97.60% zeros
mqtt.msg: 79.23% zeros
mqtt.msgtype: 96.79% zeros
mqtt.proto_len: 99.21% zeros
mqtt.protoname: 79.23% zeros
mqtt.topic: 79.23% zeros
mqtt.topic_len: 99.21% zeros
mqtt.ver: 99.21% zeros
number of 

In [7]:
y_label = df.pop("Attack_label")
y_type = df.pop('Attack_type')

In [10]:
drop_columns = [
    "frame.time",            # Date and time (Alta cardialidade)
    "ip.src_host",           # Source Host (Alta cardialidade)
    "ip.dst_host",           # Destination Host (Alta cardialidade)
    "arp.dst.proto_ipv4",    # Target IP address (Alta cardialidade)
    "arp.src.proto_ipv4",    # Sender IP address (Alta cardialidade)
    "http.file_data",        # File Data (raw content, unstructured string/bytes)
    "http.request.method",   # HTTP method (likely has little variation; use only if modeling request types)
    "http.request.full_uri", # Full request URI (very high cardinality string, likely unique per request)
    "http.request.version",  # HTTP version (may have little variance and minimal predictive value)
    "tcp.dstport",           # Destination Port (label type, likely high cardinality or redundant with `tcp.srcport`)
    "tcp.options",           # TCP Options (Alta cardialidade - sequence of bytes, not directly useful without custom parsing)
    "tcp.payload",           # TCP Payload (Alta cardialidade - Dado não estruturado)
    "tcp.srcport"            # Source Port (unsigned int, possibly high cardinality, often randomized)
]

#df.drop(drop_columns, axis=1, inplace=True)
df.shape

(157800, 19)

In [ ]:
objList = df.select_dtypes(include = "object").columns
df_quantum = df.drop(objList, axis=1)

In [18]:
from sklearn.preprocessing import MinMaxScaler

X_train, X_test, y_train, y_test = train_test_split(df_quantum, y_label, test_size=0.10, random_state=42)

X_train = np.array(X_train, requires_grad=False)
Y_train = np.array(y_train * 2 - np.ones(len(y_train)), requires_grad=False)

In [19]:
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train_scaled = scaler.fit_transform(X_train)

In [20]:
np.unique(X_train), np.unique(X_train_scaled)

(tensor([0.00000000e+00, 4.00000000e-05, 4.10000000e-05, ...,
         4.29473118e+09, 4.29473166e+09, 4.29494715e+09], shape=(161447,), requires_grad=False),
 tensor([0.00000000e+00, 6.04254968e-13, 6.19361342e-13, ...,
         3.14154472e+00, 3.14159265e+00, 3.14159265e+00], shape=(185366,), requires_grad=True))

In [21]:
np.unique(Y_train)

tensor([-1.,  1.], requires_grad=False)

Classificadores híbridos (computador clássico + circuito quântico)

VQC

In [22]:
num_qubits = X_train.shape[1]
num_layers = 3

dev = qml.device("default.qubit", wires=num_qubits)

# quantum circuit functions
def statepreparation(x):
    #qml.BasisEmbedding(x, wires=range(0, num_qubits))
    qml.AngleEmbedding(x, wires=range(num_qubits), rotation='Y')

def layer(W):

    qml.Rot(W[0, 0], W[0, 1], W[0, 2], wires=0)
    qml.Rot(W[1, 0], W[1, 1], W[1, 2], wires=1)
    qml.Rot(W[2, 0], W[2, 1], W[2, 2], wires=2)
    qml.Rot(W[3, 0], W[3, 1], W[3, 2], wires=3)

    #qml.CNOT(wires=[0, 1])QVC
    #qml.CNOT(wires=[1, 2])
    #qml.CNOT(wires=[2, 3])
    #qml.CNOT(wires=[3, 0])

    if num_qubits >= 2:
        qml.CNOT(wires=[0, 1])
    if num_qubits >= 3:
        qml.CNOT(wires=[1, 2])
    if num_qubits >= 4:
        qml.CNOT(wires=[2, 3])
    if num_qubits >= 1 and num_qubits != 0: # Ensure at least 1 qubit before this
        qml.CNOT(wires=[num_qubits - 1, 0]) # Wrap around CNOT

@qml.qnode(dev, interface="autograd")
def circuit(weights, x):

    statepreparation(x)

    for W in weights:
        layer(W)

    return qml.expval(qml.PauliZ(0))

def variational_classifier(weights, bias, x):
    return circuit(weights, x) + bias

def square_loss(labels, predictions):
    loss = 0
    for l, p in zip(labels, predictions):
        loss = loss + (l - p) ** 2

    loss = loss / len(labels)
    return loss

def accuracy(labels, predictions):

    loss = 0
    for l, p in zip(labels, predictions):
        if abs(l - p) < 1e-5:
            loss = loss + 1
    loss = loss / len(labels)

    return loss

def cost(weights, bias, X, Y):
    predictions = [variational_classifier(weights, bias, x) for x in X]
    return square_loss(Y, predictions)

Carregando modelo - Usando NumPy

In [25]:
model_dir = "./models/"
try:
    loaded_weights = np.load(os.path.join(model_dir, "vqc_6iter_7qubits_weights.npy"), allow_pickle=True)
    loaded_bias = np.load(os.path.join(model_dir, "vqc_6iter_7qubits_bias.npy"), allow_pickle=True)
    print(f"Trained model parameters loaded from '{model_dir}'")
except FileNotFoundError:
    print(f"Error: Model files not found in '{model_dir}'. Please ensure the model was saved.")
    exit() 
x_test = scaler.fit_transform(X_test)
sample_data_point = x_test[0] # Example: first data point from your full scaled dataset

prediction_value = variational_classifier(loaded_weights, loaded_bias, np.array(sample_data_point, requires_grad=False))
predicted_class = np.sign(prediction_value) # Convert to -1 or 1

print(f"\nSample data point: {sample_data_point}")
print(f"Prediction (raw output): {prediction_value.numpy()}") # Use .numpy() to convert pennylane.numpy to regular numpy
print(f"Predicted class (-1 or 1): {predicted_class.numpy()}")

# If you were doing multi-class, your prediction logic would change here
# For example:
# def multi_class_predictions(weights, bias, X_data):
#     raw_outputs = []
#     for x in X_data:
#         # assuming your circuit returns a list of expvals for multi-class
#         exp_vals = circuit(weights, x)
#         # Apply softmax or other transformation
#         probabilities = np.exp(exp_vals - np.max(exp_vals)) / np.sum(np.exp(exp_vals - np.max(exp_vals)))
#         raw_outputs.append(probabilities)
#     return np.array(raw_outputs)
#
# loaded_multi_class_predictions = multi_class_predictions(loaded_weights, loaded_bias, X_scaled_full[:5])
# print(f"Multi-class predictions for first 5 samples:\n{loaded_multi_class_predictions}")
# predicted_labels = np.argmax(loaded_multi_class_predictions, axis=1)
# print(f"Predicted labels (indices): {predicted_labels}")

Trained model parameters loaded from './models/'

Sample data point: [5.18048400e-07 1.83203169e+00 3.11426362e+00 1.78136237e+00
 2.13628300e+00 3.14159265e+00 7.23656282e-06]
Prediction (raw output): 1.3702887018268868
Predicted class (-1 or 1): 1.0


Carregando modelo - Usando Pickle

In [23]:
# --- Loading ---
model_path = "./models/vqc_6iter_7qubits.pkl"
with open(model_path, 'rb') as f:
    loaded_model_data = pickle.load(f)

loaded_weights = loaded_model_data['weights']
loaded_bias = loaded_model_data['bias']

print(f"Trained model parameters loaded from '{model_path}' using pickle.")
print(f"Hyperparameters:\nWeights: {loaded_weights}\nBias:{loaded_bias}")
# Now you can use loaded_weights and loaded_bias with your circuit definition

Trained model parameters loaded from './models/vqc_6iter_7qubits.pkl' using pickle.
Hyperparameters:
Weights: [[[ 6.42052862e-01 -1.95016371e-01  5.78372539e-01]
  [ 5.96277283e-01  4.21564990e-01  5.12470495e-01]
  [ 6.33080565e-01  3.55136976e-01  6.24534625e-01]
  [ 4.10598506e-03  1.44043571e-03  1.45427351e-02]
  [ 7.61037725e-03  1.21675016e-03  4.43863233e-03]
  [ 3.33674327e-03  1.49407907e-02 -2.05158264e-03]
  [ 3.13067702e-03 -8.54095739e-03 -2.55298982e-02]]

 [[-3.38874696e-01 -6.23271491e-01  1.57231626e-01]
  [ 1.97420656e-01  6.13341505e-01  1.04841928e-01]
  [-1.87183846e-03  1.53277921e-02  1.46935877e-02]
  [ 1.54947428e-03  3.78162521e-03 -8.87785745e-03]
  [-1.98079647e-02 -3.47912149e-03  1.56348969e-03]
  [ 1.23029068e-02  1.20237985e-02 -3.87326817e-03]
  [-3.02302751e-03 -1.04855297e-02 -1.42001794e-02]]

 [[-6.22940222e-02 -6.26381747e-01 -5.09652180e-03]
  [-4.38074301e-03 -1.25279536e-02  7.77490357e-03]
  [-1.61389785e-02 -2.12740280e-03 -8.95466560e-03]
  

In [24]:
X_test = np.array(X_test, requires_grad=False)
Y_test = np.array(y_test * 2 - np.ones(len(y_test)), requires_grad=False)

predictions = [np.sign(variational_classifier(loaded_weights, loaded_bias, x)) for x in X_test]

print(f"Predições: {predictions}")
print(f"Acurácia: {accuracy_score(Y_test, predictions)}")
print(f"Precisão: {precision_score(Y_test, predictions)}")
print(f"Recall: {recall_score(Y_test, predictions)}")
print(f"F1-Score: {f1_score(Y_test, predictions, average='macro')}")

Predições: [tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(-1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(-1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=True), tensor(1., requires_grad=T